# 03 – Análise de Autores do JMOe

## Resumo
Pipeline completo de análise de autoria do JMOe, cobrindo:

1. **Carregamento e pré-processamento** – leitura do CSV e explosão da coluna de autores.
2. **Normalização de nomes** – remoção de acentos, padronização de abreviações, mapa de unificação manual.
3. **Inferência de gênero (IBGE)** – consulta à API de nomes do IBGE para determinar gênero pelo primeiro nome.
4. **Complemento WGND** – uso do World Gender Name Dictionary para resolver nomes não encontrados no IBGE.
5. **Correções manuais** – aplicação de um mapa de correções para nomes estrangeiros ou mal formatados.
6. **Visualização** – gráfico de barras com a distribuição de gênero dos autores.

**Entrada:** `files_csv/artigos_jmoe_final.csv`  
**Saídas intermediárias e finais:** todos os CSVs são salvos em `files_csv/`


## Instalação de Dependências

In [ ]:
!pip install unidecode pandas requests matplotlib -q


## Imports

In [ ]:
import pandas as pd
import re
import ast
import os
import time
import unicodedata
import requests
import matplotlib.pyplot as plt


## 1. Carregamento dos Dados

In [ ]:
# Caminho relativo ao repositório
CAMINHO_BASE = 'files_csv/artigos_jmoe_final.csv'

df = pd.read_csv(CAMINHO_BASE)
df.head()


## 2. Pré-processamento

In [ ]:
def converter_para_lista(valor):
    """
    Converte uma string formatada como lista Python (ex: "['Autor A', 'Autor B']")
    em uma lista real. Usa ast.literal_eval, mais seguro que eval().
    Retorna lista vazia em caso de falha.
    """
    try:
        return ast.literal_eval(str(valor))
    except:
        return []

# Converte a coluna de autores de string para lista Python
df['lista_autores'] = df['autores'].apply(converter_para_lista)

# Explode: cada autor vira uma linha separada; value_counts() conta frequência
autores_frequentes = df.explode('lista_autores')['lista_autores'].value_counts()

# Cria DataFrame de autores únicos com suas contagens
df_autores = autores_frequentes.to_frame().reset_index()
df_autores.columns = ['autores', 'contagem']

print(f'Total de autores únicos identificados: {len(df_autores)}')


## 3. Normalização de Nomes

In [ ]:
def remover_acentos(texto):
    """Remove acentos usando normalização Unicode NFKD."""
    if not texto: return ''
    nfkd_form = unicodedata.normalize('NFKD', str(texto))
    return u''.join([c for c in nfkd_form if not unicodedata.combining(c)])

def normalizacao_especifica(texto):
    """
    Trata abreviações comuns em nomes brasileiros.
    Substitui 'jr' por 'junior' para não quebrar a lógica de extração do sobrenome.
    """
    texto = re.sub(r'\bjr\b\.?', 'junior', texto)
    return texto

def extrair_nomes(nome_completo):
    """
    Extrai primeiro nome e sobrenome de um nome completo.
    Suporta dois formatos:
      - 'Nome Sobrenome'   -> retorna (Nome, Sobrenome)
      - 'Sobrenome, Nome'  -> retorna (Nome, Sobrenome)
    """
    nome_completo = re.sub(' +', ' ', str(nome_completo)).strip()
    if ',' in nome_completo:
        partes = nome_completo.split(',')
        sobrenome    = partes[0].strip()
        primeiro_nome = partes[1].strip().split(' ')[0]
        return primeiro_nome, sobrenome
    partes = nome_completo.split(' ')
    if len(partes) >= 2:
        return partes[0], partes[-1]
    return nome_completo, ''

# Aplica limpeza e segmentação
df_autores['limpo'] = (
    df_autores['autores']
    .apply(remover_acentos)
    .str.lower()
    .apply(normalizacao_especifica)
)
df_autores[['primeiro_nome', 'ultimo_nome']] = df_autores['limpo'].apply(
    lambda x: pd.Series(extrair_nomes(x))
)
df_autores.head()


## 4. Mapa Mestre de Unificação de Nomes

Dicionário de correspondência para padronizar variações de escrita de um mesmo autor (abreviações vs. nome completo, erros de digitação etc.).

In [ ]:
# Cada chave é uma variação encontrada no CSV; o valor é a forma canônica
mapa_mestre = {
    # Grupo ab_bouari
    'bouari, abdou h. a. a.':           'bouari, abdou-halique a. a.',
    # Grupo ad_d\'assuncao
    "d\u2019assuncao, adaildo g.":       "d\u2019assuncao, adaildo gomes",
    # Grupo ad_guimaraes
    'guimaraes, adller o.':             'guimaraes, adller de o.',
    # Grupo ad_herbster
    'herbster, adolfo f.':              'herbster, adolfo fernandes',
    # Grupo al_serres
    'serres, alexandre j. r.':          'serres, alexandre jean rene',
    'serres, alexandre jean r.':        'serres, alexandre jean rene',
    # Grupo am_barboza
    'barboza, amanda g.':              'barboza, amanda gomes',
    # Inclua aqui demais entradas do mapa_mestre original...
    # (ver arquivo files_csv/mapa_mestre_completo.csv para a lista completa)
}

# Normaliza as chaves (minúsculo, sem espaços extras)
mapa_normalizado = {str(k).strip().lower(): v.strip() for k,v in mapa_mestre.items()}


In [ ]:
def aplicar_mapa_blindado(nome):
    """
    Busca o nome no mapa de unificação ignorando aspas e espaços residuais.
    Retorna a forma canônica se encontrado, ou o nome original caso contrário.
    """
    if not isinstance(nome, str): return nome
    nome_busca = nome.replace('"','').replace("'",'').strip().lower()
    return mapa_normalizado.get(nome_busca, nome)

# Aplica o mapa e agrupa contagens de nomes unificados
df_autores['autor_unificado'] = df_autores['limpo'].apply(aplicar_mapa_blindado)
df_final = df_autores.groupby('autor_unificado')['contagem'].sum().reset_index()
df_final.columns = ['limpo', 'contagem']
print(f'Autores únicos após agrupamento: {len(df_final)}')


## 5. Preparação para Inferência de Gênero

In [ ]:
def extrair_primeiro_nome(nome_completo):
    """
    Extrai o primeiro nome real, ignorando iniciais pontuadas (J., F.).
    Suporta formato 'Sobrenome, Nome' e 'Nome Sobrenome'.
    """
    if not isinstance(nome_completo, str): return ''
    nome_limpo = nome_completo.replace('"','').replace("'",'').strip()
    if ',' in nome_limpo:
        partes = nome_limpo.split(',')
        nomes_proprios = partes[1].strip().split()
        for n in nomes_proprios:
            token = n.replace('.','').strip()
            if len(token) > 1:  # Ignora iniciais (ex: 'J.')
                return token
        return nomes_proprios[0] if nomes_proprios else ''
    return nome_limpo.split()[0] if nome_limpo.split() else ''

df_final['primeiro_nome'] = df_final['limpo'].apply(extrair_primeiro_nome)
df_final['ultimo_nome']   = df_final['limpo'].apply(
    lambda x: str(x).split(',')[0].replace('"','').strip() if ',' in str(x) else str(x)
)
df_final = df_final.sort_values(by='contagem', ascending=False)

# Salva arquivo intermediário para revisão
CAMINHO_GENERO = 'files_csv/autores_prontos_genero.csv'
df_final.to_csv(CAMINHO_GENERO, index=False)
print(f' Arquivo salvo em: {CAMINHO_GENERO}')
display(df_final.head(15))


## 6. Inferência de Gênero via API do IBGE

In [ ]:
# Endpoint da API de nomes do IBGE (dados do Censo)
URL_IBGE = 'http://servicodados.ibge.gov.br/api/v1/censos/nomes/basica'

def inferir_genero(nome):
    """
    Consulta a API do IBGE para inferir o gênero pelo primeiro nome.
    Lógica: compara o 'rank' (popularidade) do nome nas listas masculina e feminina.
    Rank menor = nome mais comum naquele sexo.

    Retorna: 'homem', 'mulher' ou 'indefinido'.
    """
    if not nome or len(str(nome)) <= 1:
        return 'indefinido'
    nome_limpo = str(nome).strip().lower()
    try:
        r_f = requests.get(URL_IBGE, params={'nome': nome_limpo, 'sexo': 'f'}, timeout=5)
        r_m = requests.get(URL_IBGE, params={'nome': nome_limpo, 'sexo': 'm'}, timeout=5)
        res_f, res_m = r_f.json(), r_m.json()
        if not res_f and not res_m:
            return 'indefinido'
        if res_f and res_m:
            return 'mulher' if res_f[0]['rank'] < res_m[0]['rank'] else 'homem'
        return 'mulher' if res_f else 'homem'
    except Exception:
        return 'indefinido'


In [ ]:
ARQUIVO_GENERO_FINAL = 'files_csv/autores_com_genero_final.csv'

if not os.path.exists(CAMINHO_GENERO):
    print(f'Erro: {CAMINHO_GENERO} não encontrado. Rode a etapa anterior.')
else:
    df_autores = pd.read_csv(CAMINHO_GENERO)
    if 'genero' not in df_autores.columns:
        df_autores['genero'] = None  # Inicializa coluna na primeira rodada

    print('Consultando API IBGE para inferência de gênero...')

    for i, row in df_autores.iterrows():
        # Pula linhas já processadas (checkpoint)
        if pd.isna(row['genero']) or row['genero'] == 'indefinido':
            resultado = inferir_genero(row['primeiro_nome'])
            df_autores.at[i, 'genero'] = resultado
            print(f'[{i+1}/{len(df_autores)}] {str(row["primeiro_nome"]).capitalize():<12} -> {resultado}')
            # Salva checkpoint a cada 15 registros
            if i % 15 == 0:
                df_autores.to_csv(ARQUIVO_GENERO_FINAL, index=False)

    df_autores.to_csv(ARQUIVO_GENERO_FINAL, index=False)
    print(f'\n Processamento finalizado. Arquivo: {ARQUIVO_GENERO_FINAL}')


## 7. Complemento com WGND (World Gender Name Dictionary)

Resolve nomes internacionais não reconhecidos pelo IBGE.

In [ ]:
# Baixe o WGND em: https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/YPRQH8
CAMINHO_WGND   = 'files_csv/wgnd_langctry.csv'
CAMINHO_SAIDA_WGND = 'files_csv/autores_com_genero_WGND_final.csv'

if not os.path.exists(CAMINHO_WGND):
    print(f'Arquivo WGND não encontrado: {CAMINHO_WGND}')
else:
    # Carrega apenas as colunas necessárias para economizar memória
    df_wgnd = pd.read_csv(CAMINHO_WGND, usecols=['name','gender'],
                          dtype={'name':str,'gender':str})
    df_wgnd['name'] = df_wgnd['name'].str.strip().str.lower()
    df_wgnd = df_wgnd.drop_duplicates(subset=['name'])  # Um mapeamento por nome
    mapa_wgnd = dict(zip(df_wgnd['name'], df_wgnd['gender']))
    print(f'Dicionário WGND carregado: {len(mapa_wgnd)} nomes únicos.')

    def consultar_wgnd(nome, genero_atual):
        """
        Complementa a inferência do IBGE com o dicionário internacional WGND.
        Converte os códigos WGND ('M'/'F') para o padrão local ('homem'/'mulher').
        Só atua quando o gênero ainda é 'indefinido'.
        """
        if str(genero_atual).lower() not in ['indefinido','nan','none','']:
            return genero_atual  # Já definido – não sobrescreve
        genero_int = mapa_wgnd.get(str(nome).strip().lower())
        if genero_int == 'M': return 'homem'
        if genero_int == 'F': return 'mulher'
        return 'indefinido'

    df_trabalho = pd.read_csv(ARQUIVO_GENERO_FINAL)
    df_trabalho['genero'] = df_trabalho.apply(
        lambda x: consultar_wgnd(x['primeiro_nome'], x['genero']), axis=1)
    df_trabalho.to_csv(CAMINHO_SAIDA_WGND, index=False)
    print(f'✅ Salvo em: {CAMINHO_SAIDA_WGND}')
    print('\nDistribuição de gênero (IBGE + WGND):')
    print(df_trabalho['genero'].value_counts())


## 8. Correções Manuais de Gênero

Aplica um mapa de correções para nomes que as APIs não resolveram corretamente.

In [ ]:
# Mapa de correções: chave = nome 'limpo' no CSV | valor = nome completo correto
novas_correcoes = {
    'h, mouna':                'Hanachi, Mouna',
    'hasselmann, f.j.v.':      'Hasselmann, Flavio J. V.',
    'kaler, r s':              'Kaler, Rajinder S.',
    'hunagund, p v':           'Hunagund, Prabhakar V.',
    'espindola, a. a. de':     'Espindola, Aleandro A. de',
    'miranda, b.b.':           'Miranda, Breno B.',
    'bergmann, j.r.':          'Bergmann, Jose R.',
    # Adicione demais correções aqui conforme necessário
}
mapa_correcao_normalizado = {k.strip().lower(): v for k,v in novas_correcoes.items()}


In [ ]:
ARQUIVO_POS_IBGE = 'files_csv/autores_pos_ibge_final.csv'

if not os.path.exists(ARQUIVO_POS_IBGE):
    print(f'Arquivo não encontrado: {ARQUIVO_POS_IBGE}')
else:
    df = pd.read_csv(ARQUIVO_POS_IBGE)
    contador = 0
    for i in range(len(df)):
        nome_no_csv = str(df.loc[i,'limpo']).strip().lower()
        if nome_no_csv in mapa_correcao_normalizado:
            novo_nome = mapa_correcao_normalizado[nome_no_csv]
            df.at[i,'limpo']         = novo_nome
            df.at[i,'primeiro_nome'] = extrair_primeiro_nome(novo_nome)
            df.at[i,'genero']        = 'indefinido'  # Reseta para re-inferir
            contador += 1
    df.to_csv(ARQUIVO_POS_IBGE, index=False)
    print(f' {contador} registros corrigidos manualmente.')


## 9. Visualização – Distribuição de Gênero

In [ ]:
# Carrega a base final (após todos os processos de inferência e correção)
ARQUIVO_FINAL_TOTAL = 'files_csv/base_autores_FINAL_TOTAL.csv'
df_autores_viz = pd.read_csv(ARQUIVO_FINAL_TOTAL)

# Tradução dos rótulos para inglês (padrão de publicação)
traducao = {'Homem': 'Male', 'Mulher': 'Female', 'Indefinido': 'Undefined'}
df_autores_viz['genero_en'] = df_autores_viz['genero'].map(traducao)

plt.figure(figsize=(10, 4))
counts   = df_autores_viz['genero_en'].value_counts()
barplot  = counts.plot.bar(color=['#3498db', '#e74c3c', '#95a5a6'])
barplot.bar_label(barplot.containers[0], padding=3, fontsize=12)

tamanho_fonte = 14
plt.title('Gender Distribution of Authors', fontsize=tamanho_fonte + 2)
plt.ylabel('Number of Authors',             fontsize=tamanho_fonte)
plt.xlabel('Gender',                        fontsize=tamanho_fonte)
plt.xticks(rotation=0, fontsize=tamanho_fonte)
plt.yticks(fontsize=tamanho_fonte)

# Remove bordas para estética de artigo científico
for spine in barplot.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.savefig('figures/grafico_genero.png', dpi=300, bbox_inches='tight')
plt.show()
